# CVライブラリ：DETR・SAMのColab動作確認

このノートブックは、固定版のライブラリとモデルを使って物体検出・領域分割を実際に動かす確認用です。**Colabの「ランタイム」→「ランタイムのタイプを変更」からT4 GPUを選び、上から順に実行**してください。

Notebook内で図形画像を作るため、個人写真や外部データをアップロードしません。モデルの取得に数百MBの通信が発生します。推論結果は精度評価ではなく、APIの実行・戻り値・GPU利用のスモークテストです。

In [ ]:
import sys
print("Python:", sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError("このライブラリの対応Pythonは3.11〜3.13です。Colabのランタイムを確認してください。")

## 1. 固定した開発版とvision依存をインストール

コードは次の完全なGit commitから取得します。

この版ではHugging Face Hub 1.xを許容します。以前に別のモデル用パッケージを入れたランタイムは使わず、新しいColabランタイムで実行してください。

In [ ]:
%pip install 'cv-data-lesson[vision] @ git+https://github.com/708san/CV_data.git@24a0708a79f2214929e94145d721c3f3a82d6ed5'

## 2. ランタイムのGPUを確認

T4以外のGPUでも試せますが、その結果はT4の動作確認には数えません。GPUが見つからない場合はランタイム設定を変えて、最初から実行し直してください。

In [ ]:
import torch
from cvlesson import vision

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPUがありません。Colabのランタイム設定でT4 GPUを選んでください。")
device_name = torch.cuda.get_device_name(0)
print("GPU:", device_name)
if "T4" not in device_name:
    print("この実行はT4上の検証ではありません。GPU名を記録してください。")
print("cvlesson devices:", vision.available_devices())

## 3. テスト画像を作る

白い背景に色付きの図形を描いた画像を使います。DETRのカテゴリ判定精度を測る画像ではなく、モデルの入出力を試すための入力です。

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

image = Image.new("RGB", (512, 512), "white")
draw = ImageDraw.Draw(image)
draw.rectangle((80, 90, 430, 420), fill=(225, 235, 245), outline=(20, 70, 120), width=8)
draw.ellipse((170, 160, 340, 330), fill=(220, 70, 55), outline=(90, 20, 20), width=5)
draw.rectangle((210, 345, 300, 390), fill=(40, 140, 70))
image_path = "/content/cvlesson_smoke.png"
image.save(image_path)
display(image)

## 4. DETRで物体検出

初回は固定版のモデル重み（約167 MB）を取得します。COCOの80カテゴリの検出モデルなので、図形画像で検出結果が0件でも、それだけでAPIの失敗とは判定しません。

In [ ]:
detector = vision.load_detector(device="cuda:0")
detections = detector.predict(image_path, threshold=0.05)
print("検出件数:", len(detections))
for item in detections:
    x1, y1, x2, y2 = item.box
    assert 0 <= x1 <= x2 <= image.width
    assert 0 <= y1 <= y2 <= image.height
    assert 0.0 <= item.score <= 1.0
    print(item.label, round(item.score, 3), item.box)
print("DETR推論と戻り値の確認: OK")

## 5. SAMで枠からマスクを作る

初回はsafetensors形式の重み（約375 MB）を取得します。指定枠に対する候補マスクとスコアが返り、対象名は返りません。

SAMのスコアはマスク品質の予測値です。実際のIoUや正解確率ではなく、0〜1の範囲も保証されないため、値の範囲ではなく有限数かどうかを確認します。

In [ ]:
segmenter = vision.load_segmenter(device="cuda:0")
segmentation = segmenter.predict(image_path, box=(80, 90, 430, 420))
assert segmentation.mask.shape == (image.height, image.width)
assert str(segmentation.mask.dtype) == "bool"
from math import isfinite
assert isfinite(segmentation.score)
print("SAMの予測品質値（実IoU・確率ではありません）:", segmentation.score)
print("マスク形状:", segmentation.mask.shape)
print("マスク画素数:", int(segmentation.mask.sum()))
print("SAM推論と戻り値の確認: OK")

In [ ]:
import numpy as np

overlay = np.asarray(image).copy()
overlay[segmentation.mask] = (
    0.55 * overlay[segmentation.mask] + 0.45 * np.array([40, 180, 80])
).astype(np.uint8)
display(Image.fromarray(overlay))

## 6. 検出枠をSAMに渡す（任意）

DETRの検出があった場合、その先頭の枠をSAMへ渡します。検出ラベルはDETR、マスクはSAMの出力です。

In [ ]:
if detections:
    combined = segmenter.segment_detections(image_path, detections[:1])
    print(combined[0].detection.label, combined[0].segmentation.mask.shape)
    print("検出枠からSAMへの連携: OK")
else:
    print("この図形画像ではDETRの検出が0件でした。枠を指定したSAMテストは完了しています。")

## 結果の記録

- Python / PyTorchの版：上の出力
- GPU名：上の出力
- DETR：エラーなく推論し、結果型・座標を確認できたか
- SAM：元画像と同じ大きさのboolマスクが返ったか
- エラーが出た場合：最初に失敗したセルと全文を開発者へ共有

ここで確認したのは推論処理が動くことです。実データの精度や課題への適合を評価したものではありません。